In [ ]:
!pip install pymupdf sentence-transformers torch tqdm python-dotenv

In [ ]:
!pip install pytesseract

In [ ]:
!mkdir certs

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from dotenv import load_dotenv
load_dotenv()
print("[INFO] .env file refreshed.")

[INFO] .env file refreshed.


In [ ]:
from dotenv import load_dotenv
load_dotenv()
print("[INFO] .env file refreshed.")

# ingest_corpus.py
# ------------------------------------------------------------
# MINT Corpus Ingestion Script
# ------------------------------------------------------------
# What this script does:
# 1. Connects to Elasticsearch using requests
# 2. Loads NVIDIA llama-nemotron-embed-vl-1b-v2
# 3. Extracts text from each instructional PDF page
# 4. Renders each PDF page as a PNG image
# 5. Embeds each page using image + extracted text
# 6. Indexes each page into Elasticsearch
# ------------------------------------------------------------

import os
import sys
import json
import base64
import hashlib
from pathlib import Path
from typing import Dict, List, Optional, Any
from urllib.parse import quote

import requests
import fitz  # PyMuPDF
from PIL import Image
from dotenv import load_dotenv
from tqdm import tqdm

from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

load_dotenv()

CORPUS_INDEX = "math-corpus-v1"

EMBEDDING_MODEL_NAME = "nvidia/llama-nemotron-embed-vl-1b-v2"
EMBEDDING_DIMS = 2048

GDRIVE_ROOT = Path("/content/drive/MyDrive")
THESIS_DATA_DIR = GDRIVE_ROOT / "MSc AI and Big Data" / "thesis" / "data"

# PDFs are inside many topic folders under /data
PDF_ROOT_DIR = THESIS_DATA_DIR

# Rendered page images already live here
RENDER_BASE_DIR = THESIS_DATA_DIR / "rendered"

PDF_ROOT_DIR = Path(PDF_ROOT_DIR)
RENDER_BASE_DIR = Path(RENDER_BASE_DIR)

# Page render scale:
# 2.0 gives roughly 144 DPI.
# Good enough for math formulas without creating huge image files.
RENDER_ZOOM = 2.0

# Safety limit for extracted text sent along with page image.
# NVIDIA says image+text together should stay within evaluated context limits.
# For your short 1–3 page instructional PDFs, this is plenty.
MAX_TEXT_CHARS_FOR_EMBEDDING = 6000

# Use "cuda" if you have NVIDIA GPU support.
# Use "cpu" if testing without GPU.
DEVICE = os.getenv("EMBED_DEVICE", "cuda")


# ------------------------------------------------------------
# 2. Requests Elasticsearch Client
# ------------------------------------------------------------

class RequestsElasticsearchClient:
    """
    Small requests-based Elasticsearch client.

    This replaces the official elasticsearch Python client so the script can work
    more simply through tunnels such as localtunnel/ngrok/proxy URLs.
    """

    def __init__(
        self,
        host: str,
        api_key: Optional[str] = None,
        user: Optional[str] = None,
        password: Optional[str] = None,
        ca_cert: Optional[str] = None,
        verify_certs: Optional[bool] = None,
        request_timeout: int = 120,
    ):
        self.host = host.rstrip("/")
        self.request_timeout = request_timeout

        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/json",
            "Content-Type": "application/json",
        })

        if api_key:
            self.session.headers.update({
                "Authorization": f"ApiKey {api_key}"
            })
        elif user and password:
            self.session.auth = (user, password)
        else:
            raise RuntimeError(
                "No Elasticsearch auth provided. Add ELASTIC_API_KEY or "
                "ELASTIC_USER + ELASTIC_PASSWORD to .env."
            )

        if verify_certs is not None:
            self.verify = verify_certs
        elif ca_cert and Path(ca_cert).exists():
            self.verify = ca_cert
        else:
            # Useful for localtunnel/proxy testing where the public URL certificate
            # will not match the self-signed local Elasticsearch CA.
            self.verify = False

        if self.verify is False:
            requests.packages.urllib3.disable_warnings()
            print("[WARN] TLS certificate verification is disabled for requests.")

    def _url(self, path: str) -> str:
        if not path.startswith("/"):
            path = "/" + path
        return self.host + path

    def _request(
        self,
        method: str,
        path: str,
        *,
        json_body: Optional[dict] = None,
        data: Optional[str] = None,
        headers: Optional[dict] = None,
        expected: tuple = (200,),
        timeout: Optional[int] = None,
    ):
        url = self._url(path)
        resp = self.session.request(
            method=method,
            url=url,
            json=json_body,
            data=data,
            headers=headers,
            verify=self.verify,
            timeout=timeout or self.request_timeout,
        )

        if resp.status_code not in expected:
            body = resp.text[:2000]
            raise RuntimeError(
                f"Elasticsearch request failed: {method} {url}\n"
                f"Status: {resp.status_code}\n"
                f"Response: {body}"
            )

        if method.upper() == "HEAD":
            return resp.status_code

        if not resp.text.strip():
            return {}

        return resp.json()

    def info(self) -> dict:
        return self._request("GET", "/", expected=(200,))

    def indices_exists(self, index: str) -> bool:
        index = quote(index, safe="")
        status = self._request("HEAD", f"/{index}", expected=(200, 404))
        return status == 200

    def indices_create(self, index: str, body: dict) -> dict:
        index = quote(index, safe="")
        return self._request("PUT", f"/{index}", json_body=body, expected=(200,))

    def indices_refresh(self, index: str) -> dict:
        index = quote(index, safe=",*")
        return self._request("POST", f"/{index}/_refresh", expected=(200,))

    def exists(self, index: str, id: str) -> bool:
        index = quote(index, safe="")
        doc_id = quote(id, safe="")
        status = self._request("HEAD", f"/{index}/_doc/{doc_id}", expected=(200, 404))
        return status == 200

    def count(self, index: str) -> dict:
        index = quote(index, safe="")
        return self._request("GET", f"/{index}/_count", expected=(200,))

    def search(
        self,
        index: str,
        size: Optional[int] = None,
        query: Optional[dict] = None,
        knn: Optional[dict] = None,
        source: Optional[Any] = None,
        source_excludes: Optional[List[str]] = None,
        body: Optional[dict] = None,
    ) -> dict:
        index = quote(index, safe="")

        if body is None:
            body = {}

        if size is not None:
            body["size"] = size

        if query is not None:
            body["query"] = query

        if knn is not None:
            body["knn"] = knn

        if source is not None:
            body["_source"] = source

        if source_excludes is not None:
            body["_source"] = {
                "excludes": source_excludes
            }

        return self._request(
            "POST",
            f"/{index}/_search",
            json_body=body,
            expected=(200,),
        )

    def index_document(self, index: str, id: str, document: dict) -> dict:
        index = quote(index, safe="")
        doc_id = quote(id, safe="")
        return self._request(
            "PUT",
            f"/{index}/_doc/{doc_id}",
            json_body=document,
            expected=(200, 201),
        )

    def bulk(self, actions: List[dict], request_timeout: int = 300):
        """
        Minimal bulk indexer compatible with the actions used in this script.

        Expected action structure:
            {
                "_index": "...",
                "_id": "...",
                "_source": {...}
            }

        Returns:
            success_count, errors
        """

        if not actions:
            return 0, []

        lines = []

        for action in actions:
            index_name = action["_index"]
            doc_id = action["_id"]
            source = action["_source"]

            lines.append(json.dumps({
                "index": {
                    "_index": index_name,
                    "_id": doc_id
                }
            }))
            lines.append(json.dumps(source))

        payload = "\n".join(lines) + "\n"

        headers = {
            "Content-Type": "application/x-ndjson",
            "Accept": "application/json",
        }

        resp = self.session.post(
            self._url("/_bulk"),
            data=payload.encode("utf-8"),
            headers=headers,
            verify=self.verify,
            timeout=request_timeout,
        )

        if resp.status_code not in (200,):
            raise RuntimeError(
                f"Bulk request failed.\n"
                f"Status: {resp.status_code}\n"
                f"Response: {resp.text[:2000]}"
            )

        data = resp.json()
        items = data.get("items", [])

        success_count = 0
        errors = []

        for item in items:
            result = item.get("index", {})
            status = result.get("status")

            if status in (200, 201):
                success_count += 1
            else:
                errors.append(result)

        return success_count, errors


def get_bool_env(name: str, default: Optional[bool] = None) -> Optional[bool]:
    """
    Reads booleans from environment variables.

    Accepts:
      true/1/yes/y/on
      false/0/no/n/off
    """

    value = os.getenv(name)

    if value is None:
        return default

    value = value.strip().lower()

    if value in ("true", "1", "yes", "y", "on"):
        return True

    if value in ("false", "0", "no", "n", "off"):
        return False

    return default


def get_elasticsearch_client() -> RequestsElasticsearchClient:
    """
    Connect to Elasticsearch using requests.

    .env Option A: API key auth

        ELASTIC_HOST=https://localhost:9200
        ELASTIC_CA_CERT=./certs/http_ca.crt
        ELASTIC_API_KEY=your_encoded_api_key

    .env Option B: username/password auth

        ELASTIC_HOST=https://localhost:9200
        ELASTIC_CA_CERT=./certs/http_ca.crt
        ELASTIC_USER=elastic
        ELASTIC_PASSWORD=your_password

    For tunnels/proxies where certificate verification causes issues:

        ELASTIC_VERIFY_CERTS=false
    """

    host = os.getenv("ELASTIC_HOST", "https://localhost:9200")
    ca_cert = os.getenv("ELASTIC_CA_CERT")
    api_key = os.getenv("ELASTIC_API_KEY")
    user = os.getenv("ELASTIC_USER", "elastic")
    password = os.getenv("ELASTIC_PASSWORD")
    verify_certs = get_bool_env("ELASTIC_VERIFY_CERTS", default=None)

    if api_key:
        print("[INFO] Using Elasticsearch API key authentication.")
        return RequestsElasticsearchClient(
            host=host,
            ca_cert=ca_cert,
            api_key=api_key,
            verify_certs=verify_certs,
            request_timeout=120,
        )

    if not password:
        raise RuntimeError(
            "No ELASTIC_API_KEY found and ELASTIC_PASSWORD is missing. "
            "Add either ELASTIC_API_KEY or ELASTIC_USER + ELASTIC_PASSWORD to .env."
        )

    print("[INFO] Using Elasticsearch username/password authentication.")
    return RequestsElasticsearchClient(
        host=host,
        ca_cert=ca_cert,
        user=user,
        password=password,
        verify_certs=verify_certs,
        request_timeout=120,
    )


def test_elasticsearch_connection(es: RequestsElasticsearchClient) -> None:
    """
    Confirm Elasticsearch is reachable.
    """

    try:
        info = es.info()
        print(f"[OK] Connected to Elasticsearch cluster: {info['cluster_name']}")
        print(f"[OK] Elasticsearch version: {info['version']['number']}")
    except Exception as e:
        raise RuntimeError(
            "Failed to connect to Elasticsearch. "
            "Check that Elasticsearch is running/reachable through your tunnel, "
            "your password/API key is correct, and TLS verification settings are correct."
        ) from e


def create_corpus_index(es: RequestsElasticsearchClient) -> None:
    """
    Creates the corpus index if it does not already exist.

    One Elasticsearch document = one PDF page.

    This mapping matches the current ingestion source fields exactly:
    doc_id, filename, topic, domain, pdf_page, content, content_hash,
    image_path, embedding_model, embedding.
    """

    if es.indices_exists(index=CORPUS_INDEX):
        print(f"[OK] Index already exists: {CORPUS_INDEX}")
        return

    body = {
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0
        },
        "mappings": {
            "properties": {
                "doc_id": {
                    "type": "keyword"
                },
                "filename": {
                    "type": "keyword"
                },
                "topic": {
                    "type": "keyword"
                },
                "domain": {
                    "type": "keyword"
                },
                "pdf_page": {
                    "type": "integer"
                },
                "content": {
                    "type": "text"
                },
                "content_hash": {
                    "type": "keyword"
                },
                "image_path": {
                    "type": "keyword",
                    "index": False
                },
                "embedding_model": {
                    "type": "keyword"
                },
                "embedding": {
                    "type": "dense_vector",
                    "dims": EMBEDDING_DIMS,
                    "index": True,
                    "similarity": "cosine"
                }
            }
        }
    }

    es.indices_create(index=CORPUS_INDEX, body=body)
    print(f"[OK] Created index: {CORPUS_INDEX}")


# ------------------------------------------------------------
# 4. Load Nemotron Embedding Model
# ------------------------------------------------------------

def load_embedding_model() -> SentenceTransformer:
    """
    Loads NVIDIA llama-nemotron-embed-vl-1b-v2.

    First run may download several GB.
    For practical speed, use CUDA if available.
    """

    print(f"[INFO] Loading embedding model: {EMBEDDING_MODEL_NAME}")
    print(f"[INFO] Device: {DEVICE}")

    try:
        model = SentenceTransformer(
            EMBEDDING_MODEL_NAME,
            trust_remote_code=True,
            device=DEVICE
        )
    except Exception as e:
        if DEVICE == "cuda":
            print("[WARN] Failed to load on CUDA. Trying CPU instead...")
            model = SentenceTransformer(
                EMBEDDING_MODEL_NAME,
                trust_remote_code=True,
                device="cpu"
            )
        else:
            raise e

    print("[OK] Embedding model loaded.")
    return model


def test_embedding_dimension(embed_model: SentenceTransformer) -> None:
    """
    Confirms that the model outputs 2048-dimensional vectors.
    This must match the Elasticsearch dense_vector mapping.
    """

    print("[INFO] Testing embedding dimension...")

    vec = embed_model.encode_query(["test math retrieval query"])[0]
    dim = len(vec)

    if dim != EMBEDDING_DIMS:
        raise ValueError(
            f"Embedding dimension mismatch. "
            f"Expected {EMBEDDING_DIMS}, got {dim}."
        )

    print(f"[OK] Embedding dimension confirmed: {dim}")


# ------------------------------------------------------------
# 5. Embedding Helpers
# ------------------------------------------------------------

def embed_query(embed_model: SentenceTransformer, text: str) -> List[float]:
    """
    For user problem inputs at retrieval time.

    Use this later in retrieval.py, not during PDF ingestion.
    """

    text = (text or "").strip()
    vec = embed_model.encode_query([text])[0]
    return vec.tolist()


def embed_document_text(embed_model: SentenceTransformer, text: str) -> List[float]:
    """
    For text-only document chunks.

    Useful if you later want to index extracted text without page images.
    """

    text = (text or "").strip()
    vec = embed_model.encode_document([text])[0]
    return vec.tolist()


def embed_document_image_text(
    embed_model: SentenceTransformer,
    image_path: str,
    text: str = ""
) -> List[float]:
    """
    Creates a multimodal embedding from a rendered PDF page image + extracted text.

    For your PDFs, this is the correct path because:
    - normal prose is extracted as text
    - formulas may only exist visually inside the rendered page image

    NVIDIA's Sentence Transformers usage supports:
        model.encode([{"image": image, "text": text}])
    """

    image_path = str(image_path)
    text = (text or "").strip()

    # Keep text bounded so the combined image+text input does not get silly large.
    # Your instructional pages are short, so this should not cut meaningful content.
    text = text[:MAX_TEXT_CHARS_FOR_EMBEDDING]

    with Image.open(image_path) as img:
        image = img.convert("RGB")

        multimodal_doc = {
            "image": image,
            "text": text
        }

        vec = embed_model.encode([multimodal_doc])[0]

    return vec.tolist()


# ------------------------------------------------------------
# 6. PDF Text Extraction + Page Rendering
# ------------------------------------------------------------

def extract_and_render_pdf(pdf_path: str, render_dir: str) -> List[Dict]:
    """
    Extracts copyable text and renders each PDF page as a PNG image.

    Returns:
        [
            {
                "pdf_page": 1,
                "text": "...",
                "image_path": "data/rendered/file/page_001.png"
            },
            ...
        ]
    """

    pdf_path_obj = Path(pdf_path)
    render_dir_obj = Path(render_dir)
    render_dir_obj.mkdir(parents=True, exist_ok=True)

    if not pdf_path_obj.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path_obj}")

    pages = []

    doc = fitz.open(str(pdf_path_obj))

    try:
        for i, page in enumerate(doc, start=1):
            # Extract selectable/copyable text
            text = page.get_text("text").strip()

            # Render full page as image for formulas/diagrams that text extraction misses
            matrix = fitz.Matrix(RENDER_ZOOM, RENDER_ZOOM)
            pix = page.get_pixmap(matrix=matrix, alpha=False)

            image_path = render_dir_obj / f"page_{i:03d}.png"
            pix.save(str(image_path))

            pages.append({
                "pdf_page": i,
                "text": text,
                "image_path": str(image_path)
            })

    finally:
        doc.close()

    return pages


# ------------------------------------------------------------
# 7. Metadata From Filename
# ------------------------------------------------------------

def metadata_from_path(pdf_path: str) -> dict:
    """
    Derives topic and domain from folder structure.

    Expected layout:
      corpus/Set Theory/union.pdf
        -> domain = "set_theory", topic = "union"
      corpus/Fractions/fraction_to_decimal.pdf
        -> domain = "fractions", topic = "fraction_to_decimal"
    """
    pdf_path = Path(pdf_path)
    folder_name = pdf_path.parent.name          # e.g. "Set Theory"
    file_stem = pdf_path.stem                   # e.g. "union"

    domain = folder_name.lower().replace(" ", "_")   # "set_theory"
    topic  = file_stem.lower().replace(" ", "_")     # "union"

    return {"domain": domain, "topic": topic}


def clean_metadata_value(value: str) -> str:
    """
    Normalizes topic/domain text for keyword filtering.
    """

    value = (value or "").strip().lower()
    value = value.replace(" ", "_")
    return value


# ------------------------------------------------------------
# 8. Utility Hash
# ------------------------------------------------------------

def sha256_short(text: str) -> str:
    """
    Creates a short stable hash.

    Not required for indexing, but useful later for duplicate detection.
    """

    return hashlib.sha256(
        text.encode("utf-8", errors="ignore")
    ).hexdigest()[:16]


# ------------------------------------------------------------
# 9. Ingest One PDF
# ------------------------------------------------------------

def ingest_pdf(
    es: RequestsElasticsearchClient,
    embed_model: SentenceTransformer,
    pdf_path: str,
    render_base_dir: str = RENDER_BASE_DIR,
    overwrite_existing: bool = True
) -> None:
    """
    Processes one PDF:
    - derives domain/topic metadata from filename
    - extracts text
    - renders each page as image
    - embeds image + text
    - indexes each page into Elasticsearch
    """

    pdf_path_obj = Path(pdf_path)

    if not pdf_path_obj.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path_obj}")

    filename = pdf_path_obj.name
    meta = metadata_from_path(pdf_path)

    #render_dir = Path(render_base_dir) / pdf_path_obj.stem
    relative_pdf_path = pdf_path_obj.relative_to(PDF_ROOT_DIR)
    safe_render_name = "__".join(relative_pdf_path.with_suffix("").parts)
    safe_render_name = clean_metadata_value(safe_render_name)

    render_dir = Path(render_base_dir) / safe_render_name

    print(f"\n[INFO] Processing PDF: {filename}")
    print(f"[INFO] Domain: {meta['domain']}")
    print(f"[INFO] Topic: {meta['topic']}")

    pages = extract_and_render_pdf(
        pdf_path=str(pdf_path_obj),
        render_dir=str(render_dir)
    )

    if not pages:
        print(f"[WARN] No pages found in {filename}")
        return

    actions = []

    for page in tqdm(pages, desc=f"Embedding pages from {filename}"):
        text = page["text"]
        image_path = page["image_path"]
        pdf_page = page["pdf_page"]

        #doc_id = f"{pdf_path_obj.stem}-p{pdf_page:03d}"
        doc_id_base = "__".join(relative_pdf_path.with_suffix("").parts)
        doc_id_base = clean_metadata_value(doc_id_base)
        doc_id = f"{doc_id_base}-p{pdf_page:03d}"

        if not overwrite_existing:
            exists = es.exists(index=CORPUS_INDEX, id=doc_id)
            if exists:
                print(f"[SKIP] Already exists: {doc_id}")
                continue

        embedding = embed_document_image_text(
            embed_model=embed_model,
            image_path=image_path,
            text=text
        )

        content_hash = sha256_short(text + "|" + image_path)

        actions.append({
            "_index": CORPUS_INDEX,
            "_id": doc_id,
            "_source": {
                "doc_id": doc_id,
                "filename": filename,
                "topic": meta["topic"],
                "domain": meta["domain"],
                "pdf_page": pdf_page,
                "content": text,
                "content_hash": content_hash,
                "image_path": image_path,
                "embedding_model": EMBEDDING_MODEL_NAME,
                "embedding": embedding
            }
        })

    if not actions:
        print(f"[WARN] No new pages indexed from {filename}")
        return

    success, errors = es.bulk(
        actions,
        request_timeout=300,
    )

    es.indices_refresh(index=CORPUS_INDEX)

    print(f"[OK] Indexed {success} page(s) from {filename}")

    if errors:
        print(f"[WARN] Some indexing errors occurred for {filename}:")
        print(errors)


# ------------------------------------------------------------
# 10. Ingest All PDFs In Folder
# ------------------------------------------------------------

def find_pdfs_recursive(root_dir: Path, rendered_dir: Path) -> list:
    """
    Finds all PDFs under the thesis data folder.
    Skips the rendered folder.
    """

    root_dir = Path(root_dir)
    rendered_dir = Path(rendered_dir).resolve()

    pdf_files = []

    for pdf_file in root_dir.rglob("*.pdf"):
        resolved_pdf = pdf_file.resolve()

        # Skip anything inside /rendered
        try:
            resolved_pdf.relative_to(rendered_dir)
            continue
        except ValueError:
            pass

        pdf_files.append(pdf_file)

    return sorted(pdf_files)


def ingest_all_pdfs(
    es: RequestsElasticsearchClient,
    embed_model: SentenceTransformer,
    pdf_root_dir: Path = PDF_ROOT_DIR,
    render_base_dir: Path = RENDER_BASE_DIR,
    overwrite_existing: bool = True
) -> None:
    """
    Ingests every PDF inside all subfolders under /data.
    Skips /data/rendered.
    """

    pdf_root_dir = Path(pdf_root_dir)
    render_base_dir = Path(render_base_dir)

    if not pdf_root_dir.exists():
        raise FileNotFoundError(f"PDF root folder not found: {pdf_root_dir}")

    pdf_files = find_pdfs_recursive(
        root_dir=pdf_root_dir,
        rendered_dir=render_base_dir
    )

    if not pdf_files:
        print(f"[WARN] No PDF files found under: {pdf_root_dir}")
        return

    print(f"[INFO] Found {len(pdf_files)} PDF file(s) under {pdf_root_dir}")

    for pdf_file in pdf_files:
        try:
            ingest_pdf(
                es=es,
                embed_model=embed_model,
                pdf_path=str(pdf_file),
                render_base_dir=render_base_dir,
                overwrite_existing=overwrite_existing
            )
        except Exception as e:
            print(f"[ERROR] Failed to ingest {pdf_file.name}: {e}")


# ------------------------------------------------------------
# 11. Verify Index Count
# ------------------------------------------------------------

def verify_index_count(es: RequestsElasticsearchClient) -> None:
    """
    Prints the number of indexed documents in the corpus index.
    """

    es.indices_refresh(index=CORPUS_INDEX)
    count = es.count(index=CORPUS_INDEX)

    print(f"\n[OK] {CORPUS_INDEX} document count: {count['count']}")


def show_sample_documents(es: RequestsElasticsearchClient, size: int = 3) -> None:
    """
    Prints a few sample indexed documents without showing the large embedding vector.
    """

    resp = es.search(
        index=CORPUS_INDEX,
        size=size,
        query={
            "match_all": {}
        },
        source_excludes=["embedding"]
    )

    hits = resp["hits"]["hits"]

    if not hits:
        print("[WARN] No sample documents found.")
        return

    print("\n[INFO] Sample indexed documents:")

    for hit in hits:
        source = hit["_source"]

        print("-" * 60)
        print(f"ID: {hit['_id']}")
        print(f"Filename: {source.get('filename')}")
        print(f"Domain: {source.get('domain')}")
        print(f"Topic: {source.get('topic')}")
        print(f"Page: {source.get('pdf_page')}")
        print(f"Image: {source.get('image_path')}")
        preview = (source.get("content") or "")[:300].replace("\n", " ")
        print(f"Preview: {preview}")


# ------------------------------------------------------------
# 12. Optional: Small Retrieval Smoke Test
# ------------------------------------------------------------

def smoke_test_query(
    es: RequestsElasticsearchClient,
    embed_model: SentenceTransformer,
    query_text: str,
    k: int = 3,
    domain_filter: Optional[str] = None
) -> None:
    """
    Quick test to confirm that query embedding + kNN search works.

    This does not replace your full retrieval.py later.
    It is just a sanity check.
    """

    print(f"\n[INFO] Smoke test query: {query_text}")

    query_vector = embed_query(embed_model, query_text)

    knn_body = {
        "field": "embedding",
        "query_vector": query_vector,
        "k": k,
        "num_candidates": 50
    }

    if domain_filter:
        knn_body["filter"] = {
            "term": {
                "domain": clean_metadata_value(domain_filter)
            }
        }

    resp = es.search(
        index=CORPUS_INDEX,
        knn=knn_body,
        source=["doc_id", "filename", "topic", "domain", "pdf_page", "content", "image_path"]
    )

    hits = resp["hits"]["hits"]

    if not hits:
        print("[WARN] No retrieval results found.")
        return

    print("[INFO] Retrieval results:")

    for rank, hit in enumerate(hits, start=1):
        source = hit["_source"]
        preview = (source.get("content") or "")[:250].replace("\n", " ")

        print("-" * 60)
        print(f"Rank: {rank}")
        print(f"Score: {hit['_score']}")
        print(f"Doc ID: {source.get('doc_id')}")
        print(f"Filename: {source.get('filename')}")
        print(f"Domain: {source.get('domain')}")
        print(f"Topic: {source.get('topic')}")
        print(f"Page: {source.get('pdf_page')}")
        print(f"Preview: {preview}")


# ------------------------------------------------------------
# 13. Main Runner
# ------------------------------------------------------------

def main() -> None:
    print("\n========== MINT Corpus Ingestion ==========\n")

    print("[INFO] Expected folder layout:")
    print(f"       PDF root: {PDF_ROOT_DIR}")
    print(f"       Rendered: {RENDER_BASE_DIR}")
    print(f"       Index:    {CORPUS_INDEX}")
    print()

    es = get_elasticsearch_client()
    test_elasticsearch_connection(es)

    #create_corpus_index(es)

    embed_model = load_embedding_model()
    test_embedding_dimension(embed_model)

    ingest_all_pdfs(
        es=es,
        embed_model=embed_model,
        pdf_root_dir=PDF_ROOT_DIR,
        render_base_dir=RENDER_BASE_DIR,
        overwrite_existing=True
    )

    verify_index_count(es)
    show_sample_documents(es, size=3)

    # Optional sanity retrieval test.
    # Change this to match one of your actual PDF topics.
    smoke_test_query(
        es=es,
        embed_model=embed_model,
        query_text="Convert a fraction 1/3 to decimal",
        k=3
    )

    print("\n========== Ingestion Complete ==========\n")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n[STOPPED] Ingestion interrupted by user.")
        sys.exit(130)
    except Exception as e:
        print(f"\n[FATAL] {e}")
        sys.exit(1)


[INFO] .env file refreshed.

========== MINT Corpus Ingestion ==========

[INFO] Expected folder layout:
       PDF root: /content/drive/MyDrive/MSc AI and Big Data/thesis/data
       Rendered: /content/drive/MyDrive/MSc AI and Big Data/thesis/data/rendered
       Index:    math-corpus-v1

[INFO] Using Elasticsearch username/password authentication.
[WARN] TLS certificate verification is disabled for requests.
[OK] Connected to Elasticsearch cluster: elasticsearch
[OK] Elasticsearch version: 8.15.1
[INFO] Loading embedding model: nvidia/llama-nemotron-embed-vl-1b-v2
[INFO] Device: cuda


modules.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

[OK] Indexed 5 page(s) from t_truncated_cone_or_pyramid.pdf

[INFO] Processing PDF: t_truncated_cylinder_or_prism.pdf
[INFO] Domain: surface-area-volume
[INFO] Topic: t_truncated_cylinder_or_prism


Embedding pages from t_truncated_cylinder_or_prism.pdf: 100%|██████████| 5/5 [00:31<00:00,  6.35s/it]


[OK] Indexed 5 page(s) from t_truncated_cylinder_or_prism.pdf

[INFO] Processing PDF: v_volume.pdf
[INFO] Domain: surface-area-volume
[INFO] Topic: v_volume


Embedding pages from v_volume.pdf: 100%|██████████| 5/5 [00:32<00:00,  6.45s/it]


[OK] Indexed 5 page(s) from v_volume.pdf

[OK] math-corpus-v1 document count: 808

[INFO] Sample indexed documents:
------------------------------------------------------------
ID: area-perimeter__a_area_convex_polygon-p001
Filename: a_area_convex_polygon.pdf
Domain: area-perimeter
Topic: a_area_convex_polygon
Page: 1
Image: /content/drive/MyDrive/MSc AI and Big Data/thesis/data/rendered/area-perimeter__a_area_convex_polygon/page_001.png
Preview: Mathwords A to Z Index About Feedback Home / A / Area of a Convex Polygon — Formula & Examples Area of a Convex Polygon — Formula & Examples Area of a Convex Polygon The coordinates (x1, y1), (x2, y2), (x3, y3), . . . , (xn, yn) of a convex polygon are arranged in the "determinant" below. The coordi
------------------------------------------------------------
ID: area-perimeter__a_area_convex_polygon-p002
Filename: a_area_convex_polygon.pdf
Domain: area-perimeter
Topic: a_area_convex_polygon
Page: 2
Image: /content/drive/MyDrive/MSc AI and Big